# HR Policy RAG — retrieval experiments

Trying out different retrieval strategies on the HR policy corpus and checking which one actually holds up. Model comparison runs against a **5-question, category-balanced** eval set (leave, compensation, conduct, performance, recruitment) selected from data/eval/qa_dataset.py. Scored with RAGAS (faithfulness, context precision, context_recall, answer relevancy, answer correctness, judge = Groq `openai/gpt-oss-20b`) + a plain latency check.

Baseline is plain vector search — everything else has to beat it to be worth the extra complexity.

In [1]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv(override=True)

import importlib
import mlflow
import pandas as pd
from datasets import Dataset
from collections import defaultdict

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../data/eval"))

import hr_rag.config as config
import hr_rag.llm as llm_module
import hr_rag.eval_utils as eval_module
importlib.reload(config)
importlib.reload(llm_module)
importlib.reload(eval_module)

from hr_rag.config import enable_langsmith_tracing, MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME
from hr_rag.data_loading import load_policy_documents
from hr_rag.chunking import chunk_documents
from hr_rag.llm import get_llm
from hr_rag.prompts import RAG_ANSWER_PROMPT
from hr_rag.eval_utils import format_docs, measure_latency, evaluate_rag, log_to_mlflow

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from qa_dataset import dataset, QA_ITEMS

# 5-question category-balanced sample via round-robin: keeps the run cheap,
# far below free-tier rate limits, and every selected question is verified
# answerable from the corpus (leave, compensation, conduct, performance, recruitment).
SAMPLE_SIZE = 5
_idx_by_cat = defaultdict(list)
for _i, _item in enumerate(QA_ITEMS):
    _idx_by_cat[_item.category].append(_i)
# preserve config.CATEGORIES order, then append any dataset category not
# listed there (e.g. cross-policy, which spans multiple collections).
_category_order = list(config.CATEGORIES) + [
    _c for _c in _idx_by_cat if _c not in config.CATEGORIES
]
selected_indices = []
for _r in range(SAMPLE_SIZE):
    if len(selected_indices) >= SAMPLE_SIZE:
        break
    for _cat in _category_order:
        _pool = _idx_by_cat.get(_cat, [])
        if _r < len(_pool):
            selected_indices.append(_pool[_r])
            if len(selected_indices) == SAMPLE_SIZE:
                break

dataset = {
    key: [values[index] for index in selected_indices]
    for key, values in dataset.items()
}

enable_langsmith_tracing()
# silence mlflow/dagshub 'View run/experiment at ...' URL prints
import io as _io, contextlib as _ctxlib
with _ctxlib.redirect_stdout(_io.StringIO()), _ctxlib.redirect_stderr(_io.StringIO()):
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

In [2]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import ConfigDict

from hr_rag.api.core.settings import settings
from hr_rag.config import PROJECT_ROOT
from hr_rag.qdrant_store import load_category_collections


class QdrantRetriever(BaseRetriever):
    stores: dict[str, object]
    k: int = 3
    category: str | None = None
    metadata_filter: dict | None = None
    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> list[Document]:
        categories = [self.category] if self.category else list(self.stores)
        filter_values = self.metadata_filter or {}
        documents_by_category = []
        for category in categories:
            store = self.stores.get(category)
            if store is None:
                continue
            documents_by_category.append(
                store.invoke(query, limit=self.k, metadata_filter=filter_values)
            )

        if len(documents_by_category) == 1:
            return documents_by_category[0][: self.k]

        merged = []
        index = 0
        while len(merged) < self.k and any(index < len(documents) for documents in documents_by_category):
            for documents in documents_by_category:
                if index < len(documents) and len(merged) < self.k:
                    merged.append(documents[index])
            index += 1
        return merged


class QdrantVectorStore:
    def __init__(self, stores: dict[str, object]):
        self.stores = stores

    def as_retriever(self, search_kwargs: dict | None = None) -> QdrantRetriever:
        search_kwargs = search_kwargs or {}
        metadata_filter = dict(search_kwargs.get("filter") or {})
        category = metadata_filter.pop("category", None)
        return QdrantRetriever(
            stores=self.stores,
            k=search_kwargs.get("k", 3),
            category=category,
            metadata_filter=metadata_filter,
        )

    def invoke(self, question: str, k: int = 3) -> list[Document]:
        return self.as_retriever(search_kwargs={"k": k}).invoke(question)


previous_db = globals().get("db")
if previous_db and getattr(previous_db, "stores", None):
    next(iter(previous_db.stores.values())).client.close()

docs = load_policy_documents()
chunks = chunk_documents(docs)
settings.qdrant_path = str(PROJECT_ROOT / "data" / "qdrant")
db = QdrantVectorStore(load_category_collections())
if len(db.stores) == 0:
    raise RuntimeError("No Qdrant collections found. Run scripts/ingest.py first.")

llm = get_llm()
prompt = RAG_ANSWER_PROMPT

print(len(docs), "docs ->", len(chunks), "chunks")
print(len(dataset["question"]), "eval questions")
print("Qdrant collections:", ", ".join(sorted(db.stores)))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2303.98it/s]


9 docs -> 136 chunks
5 eval questions
Qdrant collections: compensation, conduct, finance, it, leave, legal, operations, performance, recruitment


results dict to collect everything as we go — dumped into a comparison table at the end

In [3]:
results = {}

def run(name, chain, get_docs_fn, retriever_type, extra_params=None):
    r = evaluate_rag(chain, get_docs_fn, dataset)
    lat = measure_latency(chain)
    log_to_mlflow(run_name=name, result=r, latency=lat, retriever_type=retriever_type, extra_params=extra_params)
    scores = r.to_pandas().mean(numeric_only=True).to_dict()
    scores["latency_s"] = lat
    results[name] = scores
    return r, lat

In [4]:
from hr_rag.retrievers.baseline import get_baseline_retriever

retriever = get_baseline_retriever(db)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(chain.invoke("What is the leave policy?"))

The Leave Management Policy outlines the types of leaves available to all full-time, part-time, and contractual employees of TechCorp India Pvt. Ltd. This policy applies to all employees across all departments, levels, and locations including Bangalore, Mumbai, Delhi NCR, Hyderabad, and Pune offices. The specific types of leaves and their details are missing from the context.


In [5]:


result, latency = run("baseline", chain, retriever.invoke, "vector-similarity")
result, latency

RAGAS judge LLM: groq/openai/gpt-oss-20b (Groq API)


({'faithfulness': 0.9333, 'context_precision': 0.4567, 'context_recall': 1.0000, 'answer_relevancy': 0.7933, 'answer_correctness': 0.6926},
 1.692)

**Evaluation config (2026-09-08):** 5-question, category-balanced eval set — leave,
compensation, conduct, performance, recruitment — sampled from `data/eval/qa_dataset.py`
via `SAMPLE_SIZE = 5`. Scored with RAGAS (faithfulness, context precision, context recall,
answer relevancy, answer correctness) using a **Groq-hosted judge** (`groq/openai/gpt-oss-20b`,
`GROQ_API_KEY`). Progress bars, deprecation warnings and retry messages are all suppressed, so
cells print only the results.


**Trade-off:** fastest and simplest option, and it's fine for direct factual lookups. Falls apart on anything with a specific number in it — embeddings blur numbers together, so it under-ranks the chunk with the exact figure in it.

## 2. Hybrid — BM25 + dense

Policy questions are full of exact numbers ("12 days", "₹50,000", "L4"). BM25 catches literal keyword/number matches that dense search alone tends to miss.

In [4]:
from hr_rag.retrievers.hybrid import get_hybrid_retriever

retriever = get_hybrid_retriever(db, chunks)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [5]:
result, latency = run("hybrid", chain, retriever.invoke, "hybrid-bm25-dense", {"bm25_weight": 0.5})
result, latency

RAGAS judge LLM: groq/openai/gpt-oss-20b (Groq API)


({'faithfulness': 1.0000, 'context_precision': nan, 'context_recall': 1.0000, 'answer_relevancy': 0.8447, 'answer_correctness': 0.7582},
 1.728)

**Trade-off:** small latency hit (running two retrievers now) but recall on numeric questions goes up noticeably. Worth it here — the production version takes this further, running a separate hybrid collection per policy category instead of one shared index (see EXP-7 and the takeaways at the end).

## 3. Query rewriting

Employees don't ask in policy-document English. "kitni chutti milti hai" needs to become something that actually matches the corpus before it hits the retriever.

In [6]:
from hr_rag.retrievers.query_rewrite import rewrite_and_retrieve

def get_docs(q):
    return rewrite_and_retrieve(db, q)

chain = (
    {"context": RunnableLambda(get_docs) | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(chain.invoke("kitni casual leave milti hai ek saal mein?"))

Casual leave entitlement is 12 days per calendar year.


In [ ]:
result, latency = run("query_rewrite", chain, get_docs, "query-rewrite")
result, latency


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

({'faithfulness': nan, 'context_precision': 0.3444, 'context_recall': 1.0000, 'answer_relevancy': 0.7596, 'answer_correctness': 0.5213},
 2.097)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

**Trade-off:** one extra LLM call before retrieval even starts, so latency goes up. Genuinely helps on the casual/Hinglish-phrased questions in the eval set, doesn't move the needle much on the formal ones.

## 4. Multi-query

Generate a few phrasings of the same question, retrieve for each, merge. More angles into the vector space, more LLM calls.

In [5]:
from hr_rag.retrievers.multi_query import get_multi_query_retriever

retriever = get_multi_query_retriever(db)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [6]:
result, latency = run("multi_query", chain, retriever.invoke, "multi-query", {"n_queries": 3})
result, latency

RAGAS judge LLM: groq/openai/gpt-oss-20b (Groq API)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



({'faithfulness': 0.9167, 'context_precision': nan, 'context_recall': nan, 'answer_relevancy': 1.0000, 'answer_correctness': 0.8312},
 4.912)

**Trade-off:** ~3x the retrieval calls plus a generation call for the extra phrasings. Recall goes up a bit but not enough to justify the cost on a corpus this small — makes more sense on a much bigger/messier document set.

## 6. Contextual compression

Run each retrieved chunk through the LLM and keep only the sentences relevant to the question, before it ever hits the answer prompt.

In [4]:
from hr_rag.retrievers.compression import get_compression_retriever

retriever = get_compression_retriever(db)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [5]:
result, latency = run("compression", chain, retriever.invoke, "contextual-compression")
result, latency

RAGAS judge LLM: groq/openai/gpt-oss-20b (Groq API)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: htt

({'faithfulness': 0.5000, 'context_precision': 1.0000, 'context_recall': 1.0000, 'answer_relevancy': 0.8312, 'answer_correctness': 0.5813},
 392.173)

**Trade-off:** an LLM call per retrieved chunk, so this is the slowest thing here by a wide margin. Cleaner context going into the final answer, but for a 500-char chunk size the noise it's removing was already pretty small. Not worth it on this corpus.

## 7. Metadata filtering

Each chunk carries a `category` (leave / compensation / conduct / performance / recruitment). If we know which policy area a question belongs to, restrict retrieval to just that category before ranking.

In [6]:
from hr_rag.retrievers.metadata_filter import get_metadata_filtered_retriever
from qa_dataset import QA_ITEMS

question_to_category = {i.question: (i.category if i.category != "cross-policy" else None) for i in QA_ITEMS}

def get_docs(q):
    category = question_to_category.get(q)
    return get_metadata_filtered_retriever(db, category=category).invoke(q)

chain = (
    {"context": RunnableLambda(get_docs) | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [7]:
result, latency = run("metadata_filter", chain, get_docs, "metadata-filtered")
result, latency

RAGAS judge LLM: groq/openai/gpt-oss-20b (Groq API)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



({'faithfulness': nan, 'context_precision': nan, 'context_recall': nan, 'answer_relevancy': nan, 'answer_correctness': nan},
 10.758)

**Trade-off:** cuts out cross-category noise for free, no extra latency — but only works when you actually know the category up front. In the real API this is what powers RBAC: an employee role literally can't retrieve compensation-category chunks, filter is applied before ranking, not after.

## 8. HyDE

Instead of embedding the raw question, ask the LLM to write a fake policy passage that would answer it, then embed that instead. The idea being a hypothetical answer looks more like the actual corpus text than a short question does.

In [ ]:
from hr_rag.retrievers.hyde import hyde_retrieve

def get_docs(q):
    return hyde_retrieve(db, q)

chain = (
    {"context": RunnableLambda(get_docs) | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



In [9]:
result, latency = run("hyde", chain, get_docs, "hyde")
result, latency


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

({'faithfulness': nan, 'context_precision': nan, 'context_recall': nan, 'answer_relevancy': nan, 'answer_correctness': nan},
 12.535)

**Trade-off:** interesting idea, one more LLM call before retrieval. On this corpus it didn't clearly beat plain hybrid — probably because the policy chunks are short and formulaic enough that a hallucinated passage doesn't add much signal over the real question. Might matter more on longer, denser documents.

## 9. Cross-encoder rerank

Same idea as FlashRank (wide net, then rerank) but with a proper cross-encoder that scores query+doc jointly instead of a lightweight approximation.

In [1]:
from hr_rag.retrievers.cross_encoder import cross_encoder_retrieve

def get_docs(q):
    return cross_encoder_retrieve(db, q)

chain = (
    {"context": RunnableLambda(get_docs) | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

d:\shoaib projects\hr-policy-copilot\hr-policy-copilot\.venv-2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'RunnableLambda' is not defined

In [ ]:
result, latency = run("cross_encoder", chain, get_docs, "cross-encoder-rerank", {"candidate_k": 10})
result, latency


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2353.18it/s]


RAGAS judge LLM: groq/openai/gpt-oss-20b (Groq API)


**Trade-off:** marginally more accurate than FlashRank in testing, but 3-4x slower on CPU for barely any precision gain. Would reconsider this if running on GPU, but for a CPU-hosted internal tool FlashRank wins on precision-per-millisecond.

## Comparison

In [ ]:
summary = pd.DataFrame(results).T
summary = summary[[c for c in ["faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness", "latency_s"] if c in summary.columns]]
summary.round(3).sort_values("context_precision", ascending=False)

## Takeaways

- What actually ships: **per-category hybrid retrieval on Qdrant** — each policy category gets its own collection, and BM25 plus dense retrieval are fused for better exact-number matching.
- Category routing and collection-level access control keep irrelevant or restricted policy chunks out of retrieval.
- The notebook uses a **5-question, category-balanced sample** (round-robin across categories — leave, compensation, conduct, performance, recruitment) for repeatable free-tier experiments; the full 52-question dataset remains available for a final run when quota permits.
- Query rewriting, multi-query retrieval, contextual compression, and cross-encoder reranking are comparison experiments; the final production choice is Qdrant plus per-category hybrid retrieval.